In [16]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/HOGs_SVM")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import cv2
import numpy as np
import google.colab.patches as colab_pathes
import matplotlib.pyplot as plt
from MNIST import MNIST

In [18]:
minst_train = MNIST("train-images.idx3-ubyte","train-labels.idx1-ubyte")

In [19]:
image, label = minst_train.getitem(100)

In [20]:
colab_pathes.cv2_imshow(image)
print(f"{label}:{image.shape}")

5:(28, 28, 1)


In [21]:
winSize = (28,28)
blockSize = (28,28)
blockStride = (1,1)
cellSize = (14,14)
nbins = 9
HOG = cv2.HOGDescriptor(winSize, blockSize, blockStride, cellSize, nbins)

In [22]:
n_train = len(minst_train)
print(f"{n_train}")
train_image_mat = np.zeros((n_train, 36), dtype=np.float32)
train_label_mat = np.zeros((n_train), dtype=int)

60000


In [23]:
for index in range(n_train):
  image, label = minst_train.getitem(index)
  a_hog = HOG.compute(image)
  a_hog = a_hog.reshape(1,-1)
  a_hog = a_hog.astype(np.float32)

  train_image_mat[index] = a_hog
  train_label_mat[index] = label

In [24]:
svm = cv2.ml.SVM_create()
svm.setType(cv2.ml.SVM_C_SVC)
svm.setKernel(cv2.ml.SVM_LINEAR)
svm.setTermCriteria((cv2.TERM_CRITERIA_MAX_ITER, 10000, 1e-6))

In [25]:
svm.train(train_image_mat, cv2.ml.ROW_SAMPLE, train_label_mat)

True

In [26]:
svm.save("svm.xml")

In [27]:
svm = cv2.ml.SVM_load("svm.xml")

In [28]:
minst_test = MNIST("t10k-images.idx3-ubyte","t10k-labels.idx1-ubyte")

In [29]:
n_test = len(minst_test)
n_correct = 0
n_wrong = 0
for index in range(n_test):
  image, label = minst_test.getitem(index)
  a_hog = HOG.compute(image)
  a_hog = a_hog.reshape(1,-1)
  a_hog = a_hog.astype(np.float32)

  result = svm.predict(a_hog)
  result = int(result[1])

  if result == label:
    n_correct += 1
  else:
    n_wrong += 1


/tmp/ipykernel_475/1530846434.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result = int(result[1])


In [30]:
accuracy = n_correct / (n_correct + n_wrong)
print(f"{n_correct}")
print(f"{accuracy * 100}")

9261
92.61
